<a href="https://colab.research.google.com/github/LRossi69/dio-lab-bia-do-futuro/blob/main/credit_card_fraud_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Detecção de Fraude em Transações de Cartão de Crédito

**Projeto de Machine Learning — problema de classificação altamente desbalanceado**

Este notebook constrói um fluxo completo de detecção de fraude: carregamento dos dados, exploração, engenharia de atributos, divisão estratificada, tratamento do desbalanceamento, treinamento de **Regressão Logística**, **Random Forest** e **XGBoost**, ajuste de limiar, avaliação por **precision, recall e F1**, curvas ROC/Precision-Recall e explicação com **SHAP**.

> **Regra principal:** neste problema, acurácia isolada pode ser enganosa. A classe fraude representa apenas uma pequena fração das transações, então o projeto prioriza a capacidade de encontrar fraudes sem ignorar o custo dos falsos positivos.

**Dataset:** Credit Card Fraud Detection — Machine Learning Group da ULB, disponibilizado no Kaggle. A base possui 284.807 transações, 492 fraudes (0,172%) e 28 variáveis V1–V28 transformadas por PCA, além de `Time`, `Amount` e `Class`.


## 1. Objetivos

- Entender o desbalanceamento da variável `Class`;
- Criar `Amount_log = log1p(Amount)`;
- Separar treino, validação e teste de forma estratificada;
- Padronizar as variáveis quando necessário;
- Comparar estratégias de pesos de classe, undersampling e oversampling;
- Treinar Logistic Regression, Random Forest e XGBoost;
- Avaliar com precision, recall, F1, ROC-AUC e principalmente PR-AUC;
- Ajustar o limiar de decisão usando a validação;
- Explicar o modelo final com importância das variáveis e SHAP;
- Salvar tabelas e gráficos em `results/` e `figures/`.


## 2. Instalação das bibliotecas

Se estiver usando Google Colab, execute a célula abaixo. Em um ambiente já configurado, ela pode ser ignorada.

O dataset **não é armazenado no repositório**. O notebook faz o download diretamente da fonte pública do Kaggle.


In [ ]:
# Se necessário, descomente:
# %pip install -q pandas numpy matplotlib seaborn scikit-learn imbalanced-learn xgboost shap requests

In [ ]:
import os
import io
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler
from xgboost import XGBClassifier
import shap

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
BASE_DIR = Path(".")
FIG_DIR = BASE_DIR / "figures"
RESULT_DIR = BASE_DIR / "results"
FIG_DIR.mkdir(exist_ok=True)
RESULT_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")


## 3. Coleta dos dados

A fonte original informa que a base contém transações de cartões de portadores europeus realizadas em dois dias de setembro de 2013. Os campos V1–V28 são componentes principais anonimizados; `Time` e `Amount` não passaram por essa transformação.

O endereço abaixo é usado apenas para baixar a base. O arquivo `creditcard.csv` não deve ser versionado no GitHub.


In [ ]:
import requests

DATA_URL = "https://www.kaggle.com/api/v1/datasets/download/mlg-ulb/creditcardfraud"
DATA_DIR = BASE_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)
ZIP_PATH = DATA_DIR / "creditcardfraud.zip"
CSV_PATH = DATA_DIR / "creditcard.csv"

if not CSV_PATH.exists():
    print("Baixando dataset...")
    response = requests.get(DATA_URL, timeout=120)
    response.raise_for_status()
    ZIP_PATH.write_bytes(response.content)

    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        csv_names = [name for name in z.namelist() if name.endswith("creditcard.csv")]
        if not csv_names:
            raise FileNotFoundError("creditcard.csv não encontrado dentro do ZIP.")
        with z.open(csv_names[0]) as source, open(CSV_PATH, "wb") as target:
            target.write(source.read())

df = pd.read_csv(CSV_PATH)
print(f"Shape: {df.shape}")
df.head()


In [ ]:
print("Colunas:")
print(df.columns.tolist())

print("\nTipos:")
display(df.dtypes.to_frame("dtype"))

print("\nValores ausentes:")
display(df.isna().sum().sort_values(ascending=False).head(10).to_frame("missing"))

print("\nDuplicatas:", df.duplicated().sum())


## 4. Exploração do desbalanceamento

A documentação do dataset informa 492 fraudes em 284.807 transações, aproximadamente 0,172%. Esse cenário torna possível obter uma acurácia muito alta simplesmente classificando quase tudo como normal.

Por isso, a pergunta operacional passa a ser: **quantas fraudes conseguimos encontrar e quantas das transações sinalizadas realmente são fraude?**


In [ ]:
class_counts = df["Class"].value_counts().sort_index()
class_pct = df["Class"].value_counts(normalize=True).sort_index() * 100

distribution = pd.DataFrame({
    "quantidade": class_counts,
    "percentual": class_pct
})
distribution.index = ["Normal (0)", "Fraude (1)"]
display(distribution)

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x=["Normal", "Fraude"], y=class_counts.values, ax=ax)
ax.set_title("Distribuição das classes")
ax.set_ylabel("Transações")
plt.tight_layout()
plt.savefig(FIG_DIR / "class_distribution.png", dpi=160)
plt.show()


In [ ]:
fraud_rate = df["Class"].mean()
print(f"Taxa de fraude: {fraud_rate:.4%}")

majority_baseline_accuracy = (df["Class"] == 0).mean()
print(f"Acurácia de um modelo que prevê sempre 'normal': {majority_baseline_accuracy:.4%}")
print("Recall de fraude desse baseline: 0.00%")


## 5. Engenharia de atributos e separação dos dados

Criamos `Amount_log` com `log1p`, reduzindo a assimetria do valor das transações.

A separação é feita em três partes:
- **Treino:** usado para ajustar os modelos;
- **Validação:** usado para comparar estratégias e escolher o limiar;
- **Teste:** reservado para a avaliação final.

A estratificação mantém a proporção da classe fraude nas divisões. Nenhum método de undersampling/oversampling é aplicado antes da divisão, evitando vazamento de informação do teste para o treino.


In [ ]:
df_model = df.copy()
df_model["Amount_log"] = np.log1p(df_model["Amount"])

X = df_model.drop(columns="Class")
y = df_model["Class"]

# 60% treino, 20% validação, 20% teste
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.40,
    stratify=y,
    random_state=RANDOM_STATE
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_STATE
)

print("Treino:", X_train.shape, "fraudes:", int(y_train.sum()), f"({y_train.mean():.4%})")
print("Validação:", X_valid.shape, "fraudes:", int(y_valid.sum()), f"({y_valid.mean():.4%})")
print("Teste:", X_test.shape, "fraudes:", int(y_test.sum()), f"({y_test.mean():.4%})")


### Padronização

A `StandardScaler` será usada dentro do pipeline da Regressão Logística. Assim, os parâmetros de escala são aprendidos apenas com o treino. Para Random Forest e XGBoost, a padronização não é necessária para o funcionamento das árvores, então esses modelos recebem os dados sem essa etapa.


## 6. Estratégias de desbalanceamento

Serão comparadas três ideias:

1. **Pesos de classe:** dar maior peso à classe fraude durante o treinamento;
2. **Undersampling:** reduzir exemplos da classe majoritária;
3. **Oversampling:** aumentar a classe minoritária por amostragem com reposição.

O `imbalanced-learn` fornece samplers específicos para essas tarefas. O teste é feito apenas dentro do pipeline de treinamento.


In [ ]:
negative = (y_train == 0).sum()
positive = (y_train == 1).sum()
scale_pos_weight = negative / positive

print(f"Negativas no treino: {negative:,}")
print(f"Fraudes no treino:   {positive:,}")
print(f"scale_pos_weight:    {scale_pos_weight:.2f}")


## 7. Modelos

### 7.1 Regressão Logística — baseline
Um modelo linear simples fornece uma referência interpretável.

### 7.2 Random Forest
Modelo baseado em várias árvores, capaz de capturar relações não lineares.

### 7.3 XGBoost
Modelo de boosting baseado em árvores. `scale_pos_weight` será usado para dar maior peso à classe positiva.

Os parâmetros abaixo são deliberadamente moderados para manter o notebook executável em ambiente comum. A seção de evolução mostra como ampliar o `GridSearchCV`.


In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=RANDOM_STATE
        ))
    ]),

    "Random Forest": RandomForestClassifier(
        n_estimators=250,
        max_depth=None,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=RANDOM_STATE
    ),

    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.08,
        subsample=0.85,
        colsample_bytree=0.85,
        min_child_weight=3,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight,
        n_jobs=-1,
        random_state=RANDOM_STATE
    )
}


In [ ]:
def evaluate_model(name, model, X_eval, y_eval, threshold=0.5):
    proba = model.predict_proba(X_eval)[:, 1]
    pred = (proba >= threshold).astype(int)

    return {
        "modelo": name,
        "limiar": threshold,
        "precision_fraude": precision_score(y_eval, pred, zero_division=0),
        "recall_fraude": recall_score(y_eval, pred, zero_division=0),
        "f1_fraude": f1_score(y_eval, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_eval, proba),
        "pr_auc": average_precision_score(y_eval, proba),
        "fraudes_sinalizadas": int(pred.sum())
    }, proba, pred


In [ ]:
fitted_models = {}
validation_results = []
validation_proba = {}
validation_pred = {}

for name, model in models.items():
    print(f"Treinando: {name}")
    model.fit(X_train, y_train)
    fitted_models[name] = model

    result, proba, pred = evaluate_model(
        name, model, X_valid, y_valid, threshold=0.5
    )
    validation_results.append(result)
    validation_proba[name] = proba
    validation_pred[name] = pred

validation_df = pd.DataFrame(validation_results).sort_values(
    "f1_fraude", ascending=False
)
display(validation_df)


## 8. Comparação de undersampling e oversampling

Para cumprir a regra de testar as alternativas, usamos a Regressão Logística com:
- `RandomUnderSampler`;
- `RandomOverSampler`.

O objetivo não é declarar previamente que uma técnica é melhor. A comparação será feita pelas métricas de fraude no conjunto de validação.


In [ ]:
sampling_models = {
    "LogReg + UnderSampling": ImbPipeline([
        ("scaler", StandardScaler()),
        ("sampler", RandomUnderSampler(
            sampling_strategy=0.5,
            random_state=RANDOM_STATE
        )),
        ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]),
    "LogReg + OverSampling": ImbPipeline([
        ("scaler", StandardScaler()),
        ("sampler", RandomOverSampler(
            sampling_strategy=0.5,
            random_state=RANDOM_STATE
        )),
        ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ])
}

sampling_results = []
sampling_fitted = {}

for name, model in sampling_models.items():
    print(f"Treinando: {name}")
    model.fit(X_train, y_train)
    sampling_fitted[name] = model
    result, _, _ = evaluate_model(name, model, X_valid, y_valid, threshold=0.5)
    sampling_results.append(result)

sampling_df = pd.DataFrame(sampling_results)
display(sampling_df.sort_values("f1_fraude", ascending=False))


## 9. Curvas ROC e Precision-Recall

A ROC-AUC é útil para medir a capacidade de separação global. Entretanto, com uma classe positiva extremamente rara, a **Precision-Recall Curve** e seu **Average Precision / PR-AUC** ajudam a visualizar melhor o compromisso entre encontrar fraudes e gerar alertas falsos.

O recall continua sendo uma métrica central neste projeto porque representa a proporção das fraudes reais que foram detectadas.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for name, proba in validation_proba.items():
    fpr, tpr, _ = roc_curve(y_valid, proba)
    auc = roc_auc_score(y_valid, proba)
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.4f})")

ax.plot([0, 1], [0, 1], linestyle="--", label="Aleatório")
ax.set_title("Curva ROC — validação")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate / Recall")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "roc_validation.png", dpi=160)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for name, proba in validation_proba.items():
    precision, recall, _ = precision_recall_curve(y_valid, proba)
    ap = average_precision_score(y_valid, proba)
    ax.plot(recall, precision, label=f"{name} (AP={ap:.4f})")

ax.axhline(y=y_valid.mean(), linestyle="--", label="Prevalência da fraude")
ax.set_title("Curva Precision-Recall — validação")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "precision_recall_validation.png", dpi=160)
plt.show()


## 10. Ajuste do limiar de decisão

O padrão `0.5` não precisa ser o melhor limiar para fraude.

Aqui escolhemos o limiar na **validação**. Para uma regra objetiva e reproduzível, vamos escolher o limiar que maximiza o **F1 da classe fraude**. Depois disso, o limiar fica congelado e é usado uma única vez no teste final.

Em um projeto empresarial, o limiar deveria ser definido também a partir do custo de falso negativo, falso positivo, capacidade operacional de investigação e nível de serviço.


In [ ]:
def threshold_table(y_true, proba, thresholds=None):
    if thresholds is None:
        thresholds = np.arange(0.05, 0.951, 0.01)

    rows = []
    for threshold in thresholds:
        pred = (proba >= threshold).astype(int)
        rows.append({
            "threshold": threshold,
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred, zero_division=0),
            "f1": f1_score(y_true, pred, zero_division=0),
            "alerts": int(pred.sum())
        })
    return pd.DataFrame(rows)

threshold_results = {}

for name, proba in validation_proba.items():
    table = threshold_table(y_valid, proba)
    threshold_results[name] = table

threshold_summary = pd.DataFrame([
    {
        "modelo": name,
        **table.loc[table["f1"].idxmax()].to_dict()
    }
    for name, table in threshold_results.items()
]).sort_values("f1", ascending=False)

display(threshold_summary)


In [ ]:
# Escolha automática do modelo pelo maior F1 de validação.
# Em um projeto real, este critério deve ser substituído por uma função de custo
# alinhada ao negócio quando esses custos forem conhecidos.

best_row = threshold_summary.iloc[0]
best_model_name = best_row["modelo"]
best_threshold = float(best_row["threshold"])

print(f"Modelo selecionado pela validação: {best_model_name}")
print(f"Limiar congelado para o teste: {best_threshold:.2f}")


In [ ]:
best_validation_proba = validation_proba[best_model_name]
best_threshold_curve = threshold_results[best_model_name]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(best_threshold_curve["threshold"], best_threshold_curve["precision"], label="Precision")
ax.plot(best_threshold_curve["threshold"], best_threshold_curve["recall"], label="Recall")
ax.plot(best_threshold_curve["threshold"], best_threshold_curve["f1"], label="F1")
ax.axvline(best_threshold, linestyle="--", label=f"Limiar={best_threshold:.2f}")
ax.set_title(f"Impacto do limiar — {best_model_name}")
ax.set_xlabel("Limiar")
ax.set_ylabel("Métrica")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "threshold_tuning.png", dpi=160)
plt.show()


## 11. Avaliação final no teste

Agora o conjunto de teste entra apenas uma vez na avaliação final. O limiar não é mais otimizado aqui.

A tabela mostra:
- **Precision:** entre os alertas, quantos eram fraude;
- **Recall:** entre as fraudes reais, quantas foram encontradas;
- **F1:** equilíbrio entre precision e recall;
- **ROC-AUC:** capacidade global de ordenação;
- **PR-AUC:** qualidade da ordenação considerando a classe positiva rara.


In [ ]:
test_results = []

for name, model in fitted_models.items():
    # Para comparação justa, cada modelo também é avaliado no teste com limiar 0.5.
    result, _, _ = evaluate_model(name, model, X_test, y_test, threshold=0.5)
    result["cenario"] = "teste_limiar_0.50"
    test_results.append(result)

# Resultado do modelo selecionado com o limiar ajustado
best_model = fitted_models[best_model_name]
best_test_proba = best_model.predict_proba(X_test)[:, 1]
best_test_pred = (best_test_proba >= best_threshold).astype(int)

test_tuned = {
    "modelo": best_model_name,
    "limiar": best_threshold,
    "precision_fraude": precision_score(y_test, best_test_pred, zero_division=0),
    "recall_fraude": recall_score(y_test, best_test_pred, zero_division=0),
    "f1_fraude": f1_score(y_test, best_test_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, best_test_proba),
    "pr_auc": average_precision_score(y_test, best_test_proba),
    "fraudes_sinalizadas": int(best_test_pred.sum()),
    "cenario": "teste_limiar_otimizado"
}
test_results.append(test_tuned)

test_df = pd.DataFrame(test_results)
display(test_df)

print("\nRelatório de classificação — modelo final:")
print(
    classification_report(
        y_test,
        best_test_pred,
        target_names=["Normal", "Fraude"],
        digits=4,
        zero_division=0
    )
)


In [ ]:
cm = confusion_matrix(y_test, best_test_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Normal", "Fraude"],
    yticklabels=["Normal", "Fraude"],
    ax=ax
)
ax.set_title(f"Matriz de confusão — {best_model_name}")
ax.set_xlabel("Predito")
ax.set_ylabel("Real")
plt.tight_layout()
plt.savefig(FIG_DIR / "confusion_matrix_final.png", dpi=160)
plt.show()


## 12. Importância das variáveis

Para Random Forest e XGBoost podemos observar a importância das features baseada no modelo. Isso é útil para uma visão global, mas não significa, sozinho, que uma variável causou a fraude.

Como V1–V28 são componentes PCA anonimizados, não é possível traduzir diretamente `V17`, por exemplo, em "tipo de estabelecimento" ou "localização". A explicação deve permanecer no nível das variáveis disponíveis.


In [ ]:
# Importância quando o modelo final for baseado em árvores.
if hasattr(best_model, "feature_importances_"):
    importances = pd.Series(
        best_model.feature_importances_,
        index=X_train.columns
    ).sort_values(ascending=False).head(15)

    display(importances.to_frame("importance"))

    fig, ax = plt.subplots(figsize=(9, 6))
    importances.sort_values().plot.barh(ax=ax)
    ax.set_title(f"Top 15 importâncias — {best_model_name}")
    ax.set_xlabel("Importância")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "feature_importance_final.png", dpi=160)
    plt.show()
else:
    print("O modelo final é linear; veja os coeficientes na célula seguinte.")


In [ ]:
if best_model_name == "Logistic Regression":
    lr_model = best_model.named_steps["model"]
    coefficients = pd.Series(
        lr_model.coef_[0],
        index=X_train.columns
    ).sort_values(key=np.abs, ascending=False)

    display(coefficients.head(15).to_frame("coefficient"))


## 13. SHAP — explicação global

O SHAP atribui contribuições às variáveis para explicar a saída de um modelo. Para modelos baseados em árvores, o `TreeExplainer` é apropriado.

Nesta etapa usamos uma amostra do conjunto de teste para manter o tempo de processamento controlado. O conjunto completo continua disponível para avaliação; a amostragem é apenas para a explicação visual.


In [ ]:
# SHAP para modelos de árvore.
if best_model_name in {"Random Forest", "XGBoost"}:
    shap_sample = X_test.sample(
        n=min(1000, len(X_test)),
        random_state=RANDOM_STATE
    )

    if best_model_name == "Random Forest":
        explainer = shap.TreeExplainer(best_model)
        shap_values = explainer(shap_sample)
    else:
        explainer = shap.TreeExplainer(best_model)
        shap_values = explainer(shap_sample)

    # Para classificadores binários do sklearn, algumas versões retornam
    # uma dimensão adicional para as classes.
    values = shap_values.values
    if values.ndim == 3:
        values = values[:, :, 1]

    shap.summary_plot(
        values,
        shap_sample,
        show=False
    )
    plt.tight_layout()
    plt.savefig(FIG_DIR / "shap_summary.png", dpi=160, bbox_inches="tight")
    plt.show()
else:
    print("Para o modelo linear final, use shap.Explainer com o pipeline linear se quiser uma explicação SHAP específica.")


## 14. SHAP — explicação de uma transação marcada como fraude

A próxima célula procura uma transação de teste que o modelo marcou como fraude e mostra as contribuições das variáveis.

**Como interpretar:** valores SHAP positivos empurram a saída do modelo na direção da classe fraude; valores negativos puxam na direção oposta. Isso explica a decisão do modelo, não prova causalidade no mundo real.


In [ ]:
if best_model_name in {"Random Forest", "XGBoost"}:
    fraud_candidates = X_test.index[best_test_pred == 1]

    if len(fraud_candidates) > 0:
        example_idx = fraud_candidates[0]
        example = X_test.loc[[example_idx]]

        example_explainer = shap.TreeExplainer(best_model)
        example_shap = example_explainer(example)

        example_values = example_shap.values
        if example_values.ndim == 3:
            example_values = example_values[:, :, 1]

        print(f"Índice da transação explicada: {example_idx}")
        print(f"Probabilidade estimada de fraude: {best_model.predict_proba(example)[0,1]:.4%}")
        print(f"Classe prevista no limiar {best_threshold:.2f}: FRAUDE")

        shap.plots.waterfall(
            shap.Explanation(
                values=example_values[0],
                base_values=(
                    example_shap.base_values[0, 1]
                    if np.ndim(example_shap.base_values) > 1
                    else example_shap.base_values[0]
                ),
                data=example.iloc[0].values,
                feature_names=example.columns.tolist()
            ),
            max_display=15,
            show=False
        )
        plt.tight_layout()
        plt.savefig(FIG_DIR / "shap_single_transaction.png", dpi=160, bbox_inches="tight")
        plt.show()
    else:
        print("Nenhuma transação de teste foi sinalizada como fraude com o limiar escolhido.")
else:
    print("A explicação individual desta seção foi preparada para os modelos de árvore.")


## 15. Exportação dos resultados

As tabelas principais são salvas para que o README possa referenciar resultados gerados pelo notebook. Os arquivos são pequenos e devem ser versionados no GitHub.


In [ ]:
distribution.to_csv(RESULT_DIR / "class_distribution.csv")
validation_df.to_csv(RESULT_DIR / "validation_model_comparison.csv", index=False)
sampling_df.to_csv(RESULT_DIR / "sampling_comparison.csv", index=False)
threshold_summary.to_csv(RESULT_DIR / "threshold_summary.csv", index=False)
test_df.to_csv(RESULT_DIR / "test_results.csv", index=False)

print("Arquivos salvos em:")
for path in sorted(RESULT_DIR.glob("*.csv")):
    print("-", path)


## 16. Conclusões

### O que este projeto demonstra

1. A acurácia não é suficiente quando a fraude é extremamente rara.
2. O balanceamento deve ser aplicado somente no treino.
3. O conjunto de teste deve manter a distribuição real das classes.
4. Recall mede diretamente a parcela das fraudes encontradas.
5. Precision mede a qualidade dos alertas gerados.
6. F1 resume precision e recall em uma única métrica.
7. PR-AUC é especialmente informativa em problemas de classe positiva rara.
8. O limiar de decisão é uma decisão operacional, não uma constante universal.
9. SHAP ajuda a explicar quais variáveis influenciaram uma decisão individual.
10. Como as variáveis V1–V28 são anonimizadas por PCA, as explicações não devem inventar significado de negócio para elas.

### Limitações

- Os dados são históricos e foram coletados em apenas dois dias de setembro de 2013.
- As variáveis principais estão anonimizadas.
- O dataset não representa necessariamente o comportamento atual de fraude de uma instituição.
- O limiar usado neste projeto foi escolhido por F1 na validação; uma aplicação real deveria incorporar custos e capacidade operacional.


## 17. Próximos passos

- Expandir o `GridSearchCV`/`RandomizedSearchCV`;
- Comparar mais modelos de classificação;
- Testar outros métodos de oversampling, como SMOTE;
- Criar variáveis comportamentais quando houver histórico temporal adequado;
- Avaliar estabilidade temporal e concept drift;
- Calibrar probabilidades;
- Definir limiar por custo de negócio;
- Monitorar precision, recall, PR-AUC e taxa de alertas em produção;
- Criar uma API para receber uma transação e retornar score + explicação.
